# Fink/LSST — HEALPix Skymap Statistics

This notebook produces HEALPix skymaps of Fink/LSST alerts:
- **Total statistics** : density of all alerts across the sky
- **Per-tag statistics**: `extragalactic_lt20mag_candidate`, `extragalactic_new_candidate`, `hostless_candidate`, `in_tns`, `sn_near_galaxy_candidate`, + Solar System objects (SSO)

Annotations: Galactic plane, Galactic Centre, Magellanic Clouds, Rubin/LSST Deep Drilling Fields.

---
**Dependencies**: `healpy`, `astropy`, `numpy`, `matplotlib`, `requests`, `pandas`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import healpy as hp
import requests
from astropy.coordinates import SkyCoord, Galactic
import astropy.units as u
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

print(f'healpy version : {hp.__version__}')
print(f'numpy  version : {np.__version__}')

## 1. Configuration

In [ ]:
# ---- HEALPix parameters ----
NSIDE = 64        # resolution: ~55 arcmin per pixel (NSIDE=128 -> ~27 arcmin, slower)
NPIX  = hp.nside2npix(NSIDE)
print(f'NSIDE={NSIDE}  ->  {NPIX} pixels, resolution ~{hp.nside2resol(NSIDE, arcmin=True):.1f} arcmin')

# ---- API Fink LSST ----
FINK_API = 'https://api.lsst.fink-portal.org'

# ---- Available tags ----
TAGS = [
    'extragalactic_lt20mag_candidate',
    'extragalactic_new_candidate',
    'hostless_candidate',
    'in_tns',
    'sn_near_galaxy_candidate',
]

# Max number of alerts per tag (increase for more complete statistics)
N_ALERTS_PER_TAG = 10000
N_SSO            = 10000   # Solar System objects

## 2. Astronomical annotations

In [ ]:
def galactic_plane_radec(n_points=500):
    """Return RA/Dec of the Galactic plane (b=0)."""
    l_range = np.linspace(0, 360, n_points) * u.deg
    coords  = SkyCoord(l=l_range, b=np.zeros(n_points)*u.deg, frame='galactic')
    icrs    = coords.icrs
    return icrs.ra.deg, icrs.dec.deg

def galactic_band_radec(b_deg=10, n_points=500):
    """Return RA/Dec for a given Galactic latitude b_deg."""
    l_range = np.linspace(0, 360, n_points) * u.deg
    coords  = SkyCoord(l=l_range, b=np.full(n_points, b_deg)*u.deg, frame='galactic')
    icrs    = coords.icrs
    return icrs.ra.deg, icrs.dec.deg

# ---- Astronomical landmarks ----
LANDMARKS = {
    'Galactic Centre': SkyCoord(l=0*u.deg, b=0*u.deg, frame='galactic').icrs,
    'LMC':             SkyCoord(ra=80.89*u.deg,  dec=-69.76*u.deg,  frame='icrs'),
    'SMC':             SkyCoord(ra=13.16*u.deg,  dec=-72.80*u.deg,  frame='icrs'),
}

# ---- Rubin/LSST Deep Drilling Fields (v3.3) ----
# Source: https://www.lsst.org/scientists/survey-design
DDF = {
    'COSMOS':                  SkyCoord(ra=150.1191*u.deg, dec=2.2058*u.deg,   frame='icrs'),
    'XMM-LSS':                 SkyCoord(ra=34.3900*u.deg,  dec=-4.9000*u.deg,  frame='icrs'),
    'ECDFS':                   SkyCoord(ra=53.1250*u.deg,  dec=-28.1000*u.deg, frame='icrs'),
    'EDFS-a':                  SkyCoord(ra=58.9000*u.deg,  dec=-49.3150*u.deg, frame='icrs'),
    'EDFS-b':                  SkyCoord(ra=63.6000*u.deg,  dec=-47.6000*u.deg, frame='icrs'),
    'Euclid Deep Field South': SkyCoord(ra=61.2400*u.deg,  dec=-48.4230*u.deg, frame='icrs'),
}

print('Annotations defined:')
for name, coord in {**LANDMARKS, **DDF}.items():
    print(f'  {name:35s}  RA={coord.ra.deg:.2f}  Dec={coord.dec.deg:.2f}')

## 3. Utility functions

In [ ]:
def radec_to_healpix(ra_deg, dec_deg, nside=NSIDE):
    """Convert RA/Dec (degrees) -> HEALPix pixel indices (RING ordering)."""
    theta = np.radians(90.0 - dec_deg)   # colatitude
    phi   = np.radians(ra_deg)
    return hp.ang2pix(nside, theta, phi)

def fetch_tag_alerts(tag, n=N_ALERTS_PER_TAG, columns='r:ra,r:dec,r:midpointMjdTai'):
    """Fetch alerts for a Fink tag via the REST API."""
    url = f'{FINK_API}/api/v1/tags'
    params = {'tag': tag, 'n': n, 'columns': columns, 'output-format': 'csv'}
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text))
    df.columns = [c.replace('r:', '') for c in df.columns]
    print(f'  [{tag:40s}]  {len(df):6d} alerts')
    return df

def fetch_sso_alerts(n=N_SSO):
    """Fetch SSO (Solar System Object) alerts from Fink."""
    url2 = f'{FINK_API}/api/v1/latests'
    params = {
        'class': 'Solar System candidate',
        'n': n,
        'columns': 'i:ra,i:dec,i:jd',
        'output-format': 'csv'
    }
    try:
        r = requests.get(url2, params=params, timeout=120)
        r.raise_for_status()
        df = pd.read_csv(StringIO(r.text))
        df.columns = [c.replace('i:', '') for c in df.columns]
        if 'jd' in df.columns:
            df = df.rename(columns={'jd': 'midpointMjdTai'})
        print(f'  [{"Solar System candidate":40s}]  {len(df):6d} alerts')
        return df
    except Exception as e:
        print(f'  SSO fetch failed: {e}')
        return pd.DataFrame(columns=['ra', 'dec', 'midpointMjdTai'])

def make_skymap(ra_deg, dec_deg, nside=NSIDE, label=''):
    """Build a HEALPix skymap (count per pixel) from RA/Dec arrays."""
    skymap = np.zeros(hp.nside2npix(nside))
    mask   = np.isfinite(ra_deg) & np.isfinite(dec_deg)
    pix    = radec_to_healpix(ra_deg[mask], dec_deg[mask], nside)
    np.add.at(skymap, pix, 1)
    nonzero = np.sum(skymap > 0)
    print(f'  Skymap {label:40s}: {int(skymap.sum()):7d} entries, {nonzero:5d} occupied pixels')
    return skymap

print('Utility functions defined.')

## 4. Data retrieval

In [ ]:
print('=== Fetching alerts per tag ===')
dfs = {}
for tag in TAGS:
    try:
        dfs[tag] = fetch_tag_alerts(tag)
    except Exception as e:
        print(f'  ERROR {tag}: {e}')
        dfs[tag] = pd.DataFrame(columns=['ra', 'dec', 'midpointMjdTai'])

print('\n=== Fetching SSO alerts ===')
dfs['Solar System candidate'] = fetch_sso_alerts()

print(f'\n=== Summary ===')
total_all = 0
for tag, df in dfs.items():
    print(f'  {tag:45s} : {len(df):6d} alerts')
    total_all += len(df)
print(f'  {"TOTAL":45s} : {total_all:6d} alerts')

## 5. HEALPix skymap construction

In [ ]:
print('=== Building HEALPix skymaps ===')
skymaps = {}

all_ra  = np.concatenate([df['ra'].values  for df in dfs.values() if len(df) > 0])
all_dec = np.concatenate([df['dec'].values for df in dfs.values() if len(df) > 0])
skymaps['Total'] = make_skymap(all_ra, all_dec, label='Total')

for tag, df in dfs.items():
    if len(df) > 0:
        skymaps[tag] = make_skymap(df['ra'].values, df['dec'].values, label=tag)
    else:
        skymaps[tag] = np.zeros(hp.nside2npix(NSIDE))

print('\nSkymaps built.')

## 6. Visualisation function with astronomical annotations

In [ ]:
TAG_COLORS = {
    'extragalactic_lt20mag_candidate': 'plasma',
    'extragalactic_new_candidate':     'inferno',
    'hostless_candidate':              'magma',
    'in_tns':                          'viridis',
    'sn_near_galaxy_candidate':        'cividis',
    'Solar System candidate':          'YlOrRd',
    'Total':                           'hot',
}

# --------------------------------------------------------------------------
# EMPTY_PIX: light grey for zero-count pixels.
# Replaces the default black background so that the coordinate grid
# and annotations are clearly readable.
# --------------------------------------------------------------------------
EMPTY_PIX = '#cccccc'

def _set_cmap_bad(name, bad_color=None):
    """Return a copy of colormap `name` with bad/under pixels -> EMPTY_PIX."""
    import copy, matplotlib.cm as mcm
    bc = bad_color if bad_color is not None else EMPTY_PIX
    cm = copy.copy(mcm.get_cmap(name))
    cm.set_bad(bc)
    cm.set_under(bc)
    return cm

def _add_coord_labels():
    """Overlay RA (along equator) and Dec (central meridian) tick labels."""
    lc = '#111111'
    # Dec labels on the central meridian (phi ~ 0 rad)
    for dec in (-60, -30, 30, 60):
        theta = np.radians(90 - dec)
        hp.projtext(theta, 0.02,
                    '{:+d} deg'.format(dec),
                    lonlat=False, color=lc, fontsize=7.5,
                    ha='left', va='center', fontweight='bold')
    # RA labels along the equator (theta = pi/2)
    for ra in range(0, 360, 60):
        hp.projtext(np.pi / 2, np.radians(ra),
                    '{:d} deg'.format(ra),
                    lonlat=False, color=lc, fontsize=7.5,
                    ha='center', va='bottom', fontweight='bold')

def plot_skymap_with_annotations(
    skymap, title='', cmap='hot', unit='Number of alerts',
    log_scale=True, figsize=(14, 7), coord='C',
    show_galactic_plane=True, show_landmarks=True, show_ddf=True,
    ax=None, show=True
):
    """
    Display a HEALPix skymap in Mollweide projection with astronomical annotations.

    Parameters
    ----------
    skymap : np.ndarray  HEALPix map (RING ordering).
    log_scale : bool     If True, display log10(1 + count).
    coord : str          'C' = equatorial, 'G' = Galactic.

    Notes
    -----
    DDFs are ~9 arcmin radius << NSIDE=64 pixel (~55 arcmin).
    They are shown as MARKERS ONLY, not filled HEALPix pixels.
    """
    map_to_plot = np.log10(1 + skymap) if log_scale else skymap.copy()
    # Mark empty pixels as UNSEEN -> healpy renders them with badcolor
    map_to_plot[map_to_plot == 0] = hp.UNSEEN

    base_cmap = _set_cmap_bad(cmap)

    plt.figure(figsize=figsize)
    hp.mollview(
        map_to_plot,
        coord=['C', coord] if coord != 'C' else ['C'],
        cmap=base_cmap,
        title=title,
        unit='log10(1+N)' if log_scale else unit,
        hold=False,
        margins=(0.01, 0.06, 0.01, 0.04),
        bgcolor=EMPTY_PIX,
        badcolor=EMPTY_PIX,
    )

    # ---- Coordinate grid: primary (30/60 deg) + secondary (10/30 deg) ----
    hp.graticule(dpar=30, dmer=60, alpha=0.75, color='#444444', lw=1.1, local=False)
    hp.graticule(dpar=10, dmer=30, alpha=0.25, color='#888888', lw=0.4,  local=False)

    # ---- RA / Dec axis labels ----
    _add_coord_labels()

    # ---- Galactic plane (b=0, +/-10 deg) ----
    if show_galactic_plane:
        for b_val, ls, lw, al in [(0, '-', 2.2, 0.95), (10, '--', 1.0, 0.6), (-10, '--', 1.0, 0.6)]:
            ra_gp, dec_gp = galactic_band_radec(b_val)
            idx = np.argsort(ra_gp)
            ra_s, dec_s = ra_gp[idx], dec_gp[idx]
            jumps = np.where(np.abs(np.diff(ra_s)) > 180)[0] + 1
            for seg_ra, seg_dec in zip(np.split(ra_s, jumps), np.split(dec_s, jumps)):
                hp.projplot(
                    np.radians(90 - seg_dec), np.radians(seg_ra),
                    linestyle=ls, linewidth=lw, alpha=al,
                    color='#0080ff', lonlat=False
                )

    # ---- Landmarks ----
    if show_landmarks:
        landmark_styles = {
            'Galactic Centre': dict(marker='*', color='#e6b800', s=200, zorder=5),
            'LMC':             dict(marker='o', color='#00cc44', s=100, zorder=5),
            'SMC':             dict(marker='o', color='#00cc44', s=70,  zorder=5),
        }
        for name, coord_obj in LANDMARKS.items():
            ra_r  = np.radians(coord_obj.ra.deg)
            dec_r = np.radians(90 - coord_obj.dec.deg)
            style = landmark_styles.get(name, dict(marker='x', color='#333333', s=60))
            hp.projscatter(dec_r, ra_r, lonlat=False, **style)
            hp.projtext(dec_r, ra_r, '  ' + name,
                        color=style['color'], fontsize=7,
                        fontweight='bold', lonlat=False)

    # ---- Deep Drilling Fields ----
    # NOTE: DDFs are ~9 arcmin radius << NSIDE=64 pixel (~55 arcmin).
    # These are MARKERS ONLY -- not filled HEALPix pixels.
    if show_ddf:
        for name, coord_obj in DDF.items():
            ra_r  = np.radians(coord_obj.ra.deg)
            dec_r = np.radians(90 - coord_obj.dec.deg)
            hp.projscatter(dec_r, ra_r, lonlat=False, marker='+',
                           color='white', s=60, edgecolors='#ff7700',
                           linewidths=0.5, zorder=6,alpha=0.5)
            hp.projtext(dec_r, ra_r, '  ' + name,
                        color='#ff7700', fontsize=6.5,
                        fontstyle='italic', lonlat=False)

    # ---- Legend ----
    handles = [
        mpatches.Patch(color='#0080ff', alpha=0.9, label='Galactic plane (|b|<=10 deg)'),
        plt.Line2D([0],[0], color='#e6b800', marker='*', ms=11,
                   ls='None', label='Galactic Centre'),
        plt.Line2D([0],[0], color='#00cc44', marker='o', ms=7,
                   ls='None', label='Magellanic Clouds'),
        plt.Line2D([0],[0], color='#ff7700', marker='+', ms=7,
                   ls='None', markerfacecolor='white',
                   markeredgecolor='#ff7700', label='DDF (markers only, < 1 pixel)'),
        mpatches.Patch(color=EMPTY_PIX, ec='#aaaaaa', label='Unobserved pixels'),
    ]
    plt.gca().legend(handles=handles, loc='lower right', fontsize=7,
                     framealpha=0.90, facecolor='white', edgecolor='#aaaaaa')

    n_total = int(skymap[skymap > 0].sum())
    plt.figtext(0.12, 0.02,
                'N alerts = {:,}  |  NSIDE={}  |  DDFs = markers (< pixel size)'.format(n_total, NSIDE),
                fontsize=7.5, color='#444444')
    if show:
        plt.tight_layout()
        plt.show()

print('Visualisation function defined.')

## 7. Total skymap

In [ ]:
plot_skymap_with_annotations(
    skymaps['Total'],
    title='Fink/LSST -- Total spatial distribution of alerts (equatorial projection)',
    cmap='hot',
    log_scale=True,
)

In [ ]:
# Same map in Galactic coordinates
plot_skymap_with_annotations(
    skymaps['Total'],
    title='Fink/LSST -- Total spatial distribution (Galactic projection)',
    cmap='hot',
    log_scale=True,
    coord='G',
)

## 8. Per-tag skymaps

In [ ]:
all_tags_ordered = TAGS + ['Solar System candidate']

for tag in all_tags_ordered:
    sm = skymaps.get(tag, np.zeros(NPIX))
    n  = int(sm.sum())
    if n == 0:
        print(f'No data for: {tag}')
        continue
    cmap = TAG_COLORS.get(tag, 'viridis')
    plot_skymap_with_annotations(
        sm,
        title=f'Fink/LSST -- {tag}',
        cmap=cmap,
        log_scale=True,
    )

## 9. Multi-tag comparative panel

In [ ]:
valid_tags = [t for t in all_tags_ordered if skymaps.get(t, np.zeros(1)).sum() > 0]
n_tags     = len(valid_tags)
ncols      = 2
nrows      = int(np.ceil(n_tags / ncols))

fig, _ = plt.subplots(nrows, ncols, figsize=(20, 6 * nrows))
plt.subplots_adjust(hspace=0.001, wspace=0.001)

for i, tag in enumerate(valid_tags):
    plt.subplot(nrows, ncols, i + 1)
    sm = skymaps[tag]
    map_to_plot = np.log10(1 + sm)
    map_to_plot[map_to_plot == 0] = hp.UNSEEN

    _cm = _set_cmap_bad(TAG_COLORS.get(tag, 'viridis'))
    hp.mollview(
        map_to_plot,
        cmap=_cm,
        title=f'{tag}\n(N={int(sm.sum()):,})',
        unit='log10(1+N)',
        hold=True,
        sub=(nrows, ncols, i + 1),
        margins=(0.01, 0.07, 0.01, 0.04),
        bgcolor=EMPTY_PIX,
        badcolor=EMPTY_PIX,
    )
    hp.graticule(dpar=30, dmer=60, alpha=0.70, color='#444444', lw=1.0, local=False)
    hp.graticule(dpar=10, dmer=30, alpha=0.22, color='#888888', lw=0.38, local=False)

    # Galactic plane overlay
    ra_gp, dec_gp = galactic_plane_radec(200)
    idx = np.argsort(ra_gp)
    ra_s, dec_s = ra_gp[idx], dec_gp[idx]
    jumps = np.where(np.abs(np.diff(ra_s)) > 180)[0] + 1
    for seg_ra, seg_dec in zip(np.split(ra_s, jumps), np.split(dec_s, jumps)):
        hp.projplot(np.radians(90 - seg_dec), np.radians(seg_ra),
                    '-', linewidth=1.5, alpha=0.85, color='#0080ff', lonlat=False)

    # DDFs (markers only)
    for name, coord_obj in DDF.items():
        hp.projscatter(
            np.radians(90 - coord_obj.dec.deg), np.radians(coord_obj.ra.deg),
            marker='+', color='white', s=30,
            edgecolors='#ff7700', linewidths=0.5, zorder=6, lonlat=False,alpha=0.5
        )

# Hide unused subplots
for j in range(n_tags, nrows * ncols):
    plt.subplot(nrows, ncols, j + 1)
    plt.axis('off')

plt.suptitle('Fink/LSST -- HEALPix skymaps by classification tag',
             fontsize=14, y=1.001, fontweight='bold')
plt.tight_layout()
plt.savefig('fink_healpix_multitag.pdf', bbox_inches='tight', dpi=150)
plt.savefig('fink_healpix_multitag.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figures saved: fink_healpix_multitag.pdf / .png')

## 10. Spatial overlap statistics between tags

In [ ]:
from itertools import combinations

print('=== Occupied pixels per tag ===')
occupied = {}
for tag in valid_tags:
    pix_set = set(np.where(skymaps[tag] > 0)[0])
    occupied[tag] = pix_set
    frac = len(pix_set) / NPIX * 100
    print(f'  {tag:45s}: {len(pix_set):5d} pixels  ({frac:.2f}% of sky)')

print('\n=== Overlap between tag pairs ===')
for t1, t2 in combinations(valid_tags, 2):
    inter = len(occupied[t1] & occupied[t2])
    union = len(occupied[t1] | occupied[t2])
    jaccard = inter / union if union > 0 else 0
    print(f'  {t1[:25]:25s} n {t2[:25]:25s} : IoU={jaccard:.3f}  ({inter} shared pixels)')

## 11. Galactic-coordinate density (bias check)

In [ ]:
fig, axes = plt.subplots(len(valid_tags), 1, figsize=(12, 3 * len(valid_tags)), sharex=True)
if len(valid_tags) == 1:
    axes = [axes]

b_bins = np.linspace(-90, 90, 73)  # 2.5 deg bins
b_mid  = 0.5 * (b_bins[:-1] + b_bins[1:])

for ax, tag in zip(axes, valid_tags):
    df = dfs[tag]
    if len(df) == 0:
        ax.set_title(f'{tag} -- no data')
        continue
    coords = SkyCoord(ra=df['ra'].values*u.deg, dec=df['dec'].values*u.deg, frame='icrs')
    gal    = coords.galactic
    b_vals = gal.b.deg
    counts, _ = np.histogram(b_vals, bins=b_bins)
    # Normalise by bin area (cosine correction for solid angle)
    cos_corr = np.cos(np.radians(b_mid))
    density  = counts / (cos_corr + 1e-10)
    ax.bar(b_mid, density, width=2.5, color=plt.get_cmap(TAG_COLORS.get(tag, 'viridis'))(0.6), alpha=0.8)
    ax.axvline(0, color='#0080ff', ls='--', lw=1.2, alpha=0.8, label='Galactic plane')
    ax.set_ylabel('Density\n(cos b corr.)', fontsize=8)
    ax.set_title(f'{tag}  (N={len(df):,})', fontsize=9)
    ax.legend(fontsize=7)

axes[-1].set_xlabel('Galactic latitude b (deg)', fontsize=10)
plt.suptitle('Galactic latitude distribution per tag', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fink_galactic_latitude_per_tag.pdf', bbox_inches='tight', dpi=150)
plt.show()